In [1]:
import sys
import os
import importlib
import pandas as pd
from pathlib import Path

# Xác định thư mục gốc dự án
PROJECT_ROOT = Path.cwd().parent

# Thêm thư mục gốc vào Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import Game Engine và các Agent cơ bản
from game.game_engine import GameEngine
from agents.random_agent import RandomAgent
from agents.heuristic_agent import HeuristicAgent

# --------------------------------------------------
# QUAN TRỌNG: Reload Minimax trước Alpha-Beta
# Vì AlphaBetaAgent kế thừa từ MinimaxAgent
# --------------------------------------------------
import agents.minimax_agent as minimax_module
importlib.reload(minimax_module)
MinimaxAgent = minimax_module.MinimaxAgent

import agents.alphabeta_agent as alphabeta_module
importlib.reload(alphabeta_module)
AlphaBetaAgent = alphabeta_module.AlphaBetaAgent

# Reload MCTS
import agents.mcts_agent as mcts_module
importlib.reload(mcts_module)
MCTSAgent = mcts_module.MCTSAgent

# Kiểm tra class được nạp từ đúng module
print("Import thành công!")
print("Project root:", PROJECT_ROOT)
print("Minimax file:", minimax_module.__file__)
print("Minimax __init__:", __import__("inspect").signature(MinimaxAgent.__init__))
print("Alpha-Beta kế thừa Minimax:", issubclass(AlphaBetaAgent, MinimaxAgent))


Import thành công!
Project root: D:\TTNTNC\TicTacToe_AI
Minimax file: D:\TTNTNC\TicTacToe_AI\agents\minimax_agent.py
Minimax __init__: (self, depth=4, name='Minimax Agent')
Alpha-Beta kế thừa Minimax: True


In [2]:
agent_factories = {
    "MCTS": lambda: MCTSAgent(
        simulations=500,
        exploration=2**0.5
    ),

    "Random": lambda: RandomAgent(),

    "Heuristic": lambda: HeuristicAgent(),

    "Minimax": lambda: MinimaxAgent(depth=4),

    "Alpha-Beta": lambda: AlphaBetaAgent(depth=4)
}

print("Danh sách Agent:")

for name in agent_factories:
    print("-", name)

Danh sách Agent:
- MCTS
- Random
- Heuristic
- Minimax
- Alpha-Beta


In [3]:
agent_factories = {
    "MCTS": lambda: MCTSAgent(
        simulations=500,
        exploration=2**0.5
    ),

    "Random": lambda: RandomAgent(),

    "Heuristic": lambda: HeuristicAgent(),

    "Minimax": lambda: MinimaxAgent(depth=4),

    "Alpha-Beta": lambda: AlphaBetaAgent(depth=4)
}

print("Danh sách Agent:")

for name in agent_factories:
    print("-", name)

Danh sách Agent:
- MCTS
- Random
- Heuristic
- Minimax
- Alpha-Beta


In [4]:
def get_winner(board, win_length=4):
    board_size = len(board)

    directions = [
        (0, 1),    # Ngang
        (1, 0),    # Dọc
        (1, 1),    # Chéo xuống
        (1, -1)    # Chéo lên
    ]

    for r in range(board_size):
        for c in range(board_size):

            player = board[r][c]

            if player == 0:
                continue

            for dr, dc in directions:

                cells = []

                for i in range(win_length):
                    nr = r + i * dr
                    nc = c + i * dc

                    if not (0 <= nr < board_size and
                            0 <= nc < board_size):
                        break

                    cells.append(board[nr][nc])

                if len(cells) == win_length:
                    if all(value == player for value in cells):
                        return player

    return 0


print("Hàm kiểm tra người thắng đã sẵn sàng!")

Hàm kiểm tra người thắng đã sẵn sàng!


In [5]:
def play_match(x_name, o_name):
    # Khởi tạo Game Engine
    game = GameEngine()

    # Khởi tạo hai Agent
    agent_x = agent_factories[x_name]()
    agent_o = agent_factories[o_name]()

    agents = {
        1: agent_x,
        -1: agent_o
    }

    move_count = 0

    while not game.is_terminal():
        # Xác định người chơi hiện tại
        current_player = game.state.current_player
        current_agent = agents[current_player]

        # Agent lựa chọn hành động
        action = current_agent.choose_action(game)

        # Kiểm tra nước đi hợp lệ
        legal_actions = game.legal_actions()
        if action not in legal_actions:
            agent_name = getattr(current_agent, "name", type(current_agent).__name__)
            raise ValueError(
                f"{agent_name} chọn nước đi không hợp lệ: {action}"
            )

        # Thực hiện nước đi
        game.step(action)
        move_count += 1

    # Xác định người chiến thắng
    winner = get_winner(
        game.state.board,
        game.win_length
    )

    if winner == 1:
        winner_name = x_name
    elif winner == -1:
        winner_name = o_name
    else:
        winner_name = "Draw"

    return {
        "X_Agent": x_name,
        "O_Agent": o_name,
        "Winner": winner_name,
        "Moves": move_count
    }


In [6]:
test_result = play_match("MCTS", "Random")

print("KẾT QUẢ TRẬN ĐẤU")
print("-------------------------")
print("X:", test_result["X_Agent"])
print("O:", test_result["O_Agent"])
print("Winner:", test_result["Winner"])
print("Moves:", test_result["Moves"])

KẾT QUẢ TRẬN ĐẤU
-------------------------
X: MCTS
O: Random
Winner: MCTS
Moves: 7


In [7]:
opponents = [
    "Random",
    "Heuristic",
    "Minimax",
    "Alpha-Beta"
]

match_results = []

for opponent in opponents:

    print(f"\nĐang thi đấu: MCTS vs {opponent}")

    for match_number in range(4):

        # Đổi bên sau mỗi trận
        if match_number % 2 == 0:
            x_name = "MCTS"
            o_name = opponent
        else:
            x_name = opponent
            o_name = "MCTS"

        result = play_match(x_name, o_name)

        # Xác định kết quả từ góc nhìn MCTS
        if result["Winner"] == "MCTS":
            mcts_result = "Win"

        elif result["Winner"] == "Draw":
            mcts_result = "Draw"

        else:
            mcts_result = "Loss"

        result["Opponent"] = opponent
        result["MCTS_Result"] = mcts_result
        result["Match_Number"] = match_number + 1

        match_results.append(result)

        print(
            f"Trận {match_number + 1}: "
            f"{mcts_result} | "
            f"Số lượt: {result['Moves']}"
        )

print("\nHOÀN THÀNH TOURNAMENT!")
print("Tổng số trận:", len(match_results))


Đang thi đấu: MCTS vs Random
Trận 1: Win | Số lượt: 9
Trận 2: Win | Số lượt: 10
Trận 3: Win | Số lượt: 9
Trận 4: Win | Số lượt: 8

Đang thi đấu: MCTS vs Heuristic
Trận 1: Draw | Số lượt: 16
Trận 2: Draw | Số lượt: 16
Trận 3: Draw | Số lượt: 16
Trận 4: Draw | Số lượt: 16

Đang thi đấu: MCTS vs Minimax
Trận 1: Draw | Số lượt: 16
Trận 2: Draw | Số lượt: 16
Trận 3: Draw | Số lượt: 16
Trận 4: Draw | Số lượt: 16

Đang thi đấu: MCTS vs Alpha-Beta
Trận 1: Draw | Số lượt: 16
Trận 2: Draw | Số lượt: 16
Trận 3: Draw | Số lượt: 16
Trận 4: Draw | Số lượt: 16

HOÀN THÀNH TOURNAMENT!
Tổng số trận: 16


In [8]:
df_results = pd.DataFrame(match_results)

summary = (
    df_results
    .groupby(["Opponent", "MCTS_Result"])
    .size()
    .unstack(fill_value=0)
)

# Bổ sung các cột nếu không xuất hiện trong kết quả
for column in ["Win", "Draw", "Loss"]:
    if column not in summary.columns:
        summary[column] = 0

summary = summary[["Win", "Draw", "Loss"]]

summary["Games"] = (
    summary["Win"] +
    summary["Draw"] +
    summary["Loss"]
)

summary["Score_Rate"] = (
    (summary["Win"] + 0.5 * summary["Draw"])
    / summary["Games"] * 100
)

summary["Score_Rate"] = summary["Score_Rate"].round(2)

print("KẾT QUẢ TOURNAMENT MCTS")
display(summary)

KẾT QUẢ TOURNAMENT MCTS


MCTS_Result,Win,Draw,Loss,Games,Score_Rate
Opponent,,,,,
Alpha-Beta,0,4,0,4,50.0
Heuristic,0,4,0,4,50.0
Minimax,0,4,0,4,50.0
Random,4,0,0,4,100.0


In [9]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

# Lưu chi tiết từng trận
df_results.to_csv(
    results_dir / "mcts_tournament_details.csv",
    index=False,
    encoding="utf-8-sig"
)

# Lưu bảng thống kê
summary.to_csv(
    results_dir / "mcts_tournament_summary.csv",
    encoding="utf-8-sig"
)

print("Đã lưu kết quả Tournament!")
print("Thư mục:", results_dir)

Đã lưu kết quả Tournament!
Thư mục: D:\TTNTNC\TicTacToe_AI\results
